# 01 — Embedding Analysis

Post-training analysis of iTransformerAE embeddings: geometry, PCA, clustering,
statistical validation, and baseline comparison.

**Prerequisite:** Trained model (run `make train` or `make sweep` first).

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from tcc_itransformer.config import ExperimentConfig
from tcc_itransformer.data.fred_md import load_fred_md
from tcc_itransformer.data.preprocessing import FredMDPreprocessor
from tcc_itransformer.data.dataset import WindowDataset
from tcc_itransformer.model.autoencoder import iTransformerAE
from tcc_itransformer.seed import set_global_seed

warnings.filterwarnings("ignore", category=FutureWarning)

CONFIG_PATH = "../configs/default.yaml"
MODEL_PATH = "../results/checkpoints/best_model.pt"  # adjust as needed
FIGURES_DIR = Path("../results/figures/analysis")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

config = ExperimentConfig.from_yaml(CONFIG_PATH)
set_global_seed(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config: W={config.window_size}, d_model={config.d_model}, latent={config.latent_dim}")
print(f"Device: {device}")

## 1. Load Model and Data

In [ ]:
# Load data and create splits
df = load_fred_md(config.data_path)
prep = FredMDPreprocessor()
train_df, val_df, test_df = prep.fit_transform_split(
    df, train_end=config.train_end, val_end=config.val_end
)
n_series = train_df.shape[1]

# Load model
model = iTransformerAE(
    n_series=n_series,
    window_size=config.window_size,
    d_model=config.d_model,
    n_heads=config.n_heads,
    n_layers=config.n_layers,
    latent_dim=config.latent_dim,
    dropout=config.dropout,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()
print(f"Model loaded. N_series={n_series}")

## 2. Extract Embeddings

In [ ]:
from torch.utils.data import DataLoader

def extract_embeddings(model, dataset, device, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_z = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device)
            z = model.encode(x)
            # Flatten: (B, N, L) -> (B, N*L)
            all_z.append(z.reshape(z.shape[0], -1).cpu().numpy())
    return np.concatenate(all_z, axis=0)

train_ds = WindowDataset(train_df.values, config.window_size)
val_ds = WindowDataset(val_df.values, config.window_size)
test_ds = WindowDataset(test_df.values, config.window_size)

train_emb = extract_embeddings(model, train_ds, device)
val_emb = extract_embeddings(model, val_ds, device)
test_emb = extract_embeddings(model, test_ds, device)

train_mean = train_emb.mean(axis=0)
print(f"Embedding shapes: train={train_emb.shape}, val={val_emb.shape}, test={test_emb.shape}")

## 3. Embedding Geometry

In [ ]:
from tcc_itransformer.evaluation.embedding_quality import (
    check_embedding_collapse,
    compute_effective_rank,
    compute_isotropy,
)
from tcc_itransformer.utils.viz import plot_dim_variance_heatmap

collapsed, min_var = check_embedding_collapse(train_emb)
eff_rank = compute_effective_rank(train_emb)
isotropy = compute_isotropy(train_emb)

print(f"Collapsed: {collapsed} (min variance: {min_var:.6f})")
print(f"Effective rank: {eff_rank:.2f}")
print(f"Isotropy: {isotropy:.4f}")

fig = plot_dim_variance_heatmap(train_emb, save_path=FIGURES_DIR / "dim_variance.png")
plt.show()

## 4. Adaptive PCA

In [ ]:
from tcc_itransformer.evaluation.clustering import fit_adaptive_pca, apply_pca
from tcc_itransformer.utils.viz import plot_scree

pca, n_components = fit_adaptive_pca(
    train_emb - train_mean,
    variance_threshold=config.pca_variance_threshold,
    max_components=config.n_pca_max,
)
train_pca = apply_pca(pca, train_emb - train_mean)
test_pca = apply_pca(pca, test_emb - train_mean)

print(f"PCA components: {n_components}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.4f}")

fig = plot_scree(pca, save_path=FIGURES_DIR / "embedding_pca_scree.png")
plt.show()

## 5. K Selection

In [ ]:
from tcc_itransformer.evaluation.clustering import select_k
from tcc_itransformer.evaluation.effective_sample_size import extract_non_overlapping_indices
from tcc_itransformer.utils.viz import plot_silhouette_vs_k

non_overlap_idx = extract_non_overlapping_indices(len(train_pca), config.window_size)
train_pca_no = train_pca[non_overlap_idx]

k_range = range(3, 6)
best_k, sil_scores = select_k(train_pca_no, k_range=k_range, seed=config.seed)

print(f"Best K: {best_k}")
for k, s in zip(k_range, sil_scores):
    print(f"  K={k}: silhouette={s:.4f}")

fig = plot_silhouette_vs_k(list(k_range), sil_scores, best_k=best_k,
                           save_path=FIGURES_DIR / "silhouette_vs_k.png")
plt.show()

## 6. Regime Visualization

In [ ]:
from tcc_itransformer.evaluation.clustering import fit_kmeans
from tcc_itransformer.utils.viz import plot_regime_timeline_nber, plot_cluster_scatter

kmeans = fit_kmeans(train_pca_no, best_k, seed=config.seed)
test_labels = kmeans.predict(test_pca)

# NBER recessions (approximate)
nber = [
    ("2001-03-01", "2001-11-01"),
    ("2007-12-01", "2009-06-01"),
    ("2020-02-01", "2020-04-01"),
]
nber_pd = [(pd.Timestamp(s), pd.Timestamp(e)) for s, e in nber]

# Use test dates (approximate from index)
test_dates = test_df.index[config.window_size - 1:config.window_size - 1 + len(test_labels)]

fig = plot_regime_timeline_nber(
    test_dates.values, test_labels, nber_recessions=nber_pd,
    title="Test Set Regime Timeline",
    save_path=FIGURES_DIR / "regime_timeline_nber.png",
)
plt.show()

# 2D scatter
fig = plot_cluster_scatter(
    test_pca[:, :2], test_labels,
    title="Test Embeddings (PC1 vs PC2)",
    save_path=FIGURES_DIR / "cluster_scatter_test.png",
)
plt.show()

## 7. Statistical Validation

In [ ]:
from tcc_itransformer.evaluation.statistical_tests import (
    kruskal_wallis_per_dim,
    pairwise_mann_whitney,
)
from tcc_itransformer.utils.viz import plot_statistical_results_table

kw = kruskal_wallis_per_dim(test_pca, test_labels)
print(f"KW significant dims: {kw['n_significant']}/{test_pca.shape[1]}")
print(f"Effect sizes: {kw['effect_sizes'].round(4)}")

mw = pairwise_mann_whitney(test_pca, test_labels)
print(f"\nMW pairs: {mw['pairs']}")
print(f"Significant comparisons: {(mw['p_values_corrected'] < 0.05).sum()}")

summary = {
    "KW significant dims": kw["n_significant"],
    "KW mean effect size": float(kw["effect_sizes"].mean()),
    "MW significant pairs×dims": int((mw["p_values_corrected"] < 0.05).sum()),
}
fig = plot_statistical_results_table(summary, save_path=FIGURES_DIR / "stat_summary.png")
plt.show()

## 8. Baseline Comparison

In [ ]:
from tcc_itransformer.evaluation.baselines import run_all_baselines
from tcc_itransformer.evaluation.statistical_tests import permutation_test
from tcc_itransformer.evaluation.clustering import compute_clustering_metrics
from tcc_itransformer.utils.viz import plot_baseline_comparison_bar

# Model silhouette
model_metrics = compute_clustering_metrics(test_pca, test_labels)
model_sil = model_metrics["silhouette"]

# Baselines
baselines = run_all_baselines(test_emb - train_mean, best_k, seed=config.seed)

names = ["iTransformer"] + list(baselines.keys())
sils = [model_sil] + [b["silhouette"] for b in baselines.values()]
p_vals = [1.0]  # model vs itself
for bname, bdata in baselines.items():
    p = permutation_test(
        test_pca, bdata["embeddings"],
        stat_fn=lambda a, b: compute_clustering_metrics(
            a, kmeans.predict(a))["silhouette"] - compute_clustering_metrics(
            b, fit_kmeans(b, best_k, seed=config.seed).predict(b))["silhouette"],
        n_permutations=5000,
    )
    p_vals.append(p)

print("Baseline comparison:")
for n, s, p in zip(names, sils, p_vals):
    print(f"  {n}: sil={s:.4f}, p={p:.4f}")

fig = plot_baseline_comparison_bar(names, sils, p_vals,
                                   save_path=FIGURES_DIR / "baseline_comparison.png")
plt.show()

## 9. Summary

| Metric | Value |
|--------|-------|
| Effective rank | TBD |
| Isotropy | TBD |
| PCA components | TBD |
| Best K | TBD |
| Test silhouette | TBD |
| KW significant dims | TBD |
| vs Baseline (p-value) | TBD |

*Fill in after running all cells.*